# Qwen2.5-14B LoRA fine-tune on Colab T4

Practice run on the hybrid medical SFT dataset (same data/task mix as the MI300X run), adapted for a single T4 GPU (15GB usable) via Google Drive.

**Before running:** in `Runtime > Change runtime type`, select GPU = **T4**. Then upload `data/processed/hybrid/train.jsonl` and `val.jsonl` to your Google Drive at:
`MyDrive/MedicalLLM/data/train.jsonl` and `MyDrive/MedicalLLM/data/val.jsonl`

Checkpoints and logs are written straight to Drive (`MyDrive/MedicalLLM/output/qwen14b_t4/`) so a dropped Colab session doesn't lose progress - just rerun all cells, it auto-resumes from the last checkpoint.

**Memory note:** T4 has only 15GB VRAM total, and the 14B model + LoRA alone already uses ~10GB before training even starts - this leaves very little headroom, so batch size and sequence length are kept deliberately small (batch=1, seq_len=2048) to avoid OOM. If you're on Colab's free tier (not Pro), also expect stricter session/idle limits and no guaranteed GPU allocation.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    raise RuntimeError("This notebook is meant to run on Google Colab.")
!pip install --no-deps bitsandbytes accelerate xformers peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install --no-deps unsloth

In [ ]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/MedicalLLM"
TRAIN_PATH = f"{DRIVE_ROOT}/data/train.jsonl"
VAL_PATH = f"{DRIVE_ROOT}/data/val.jsonl"
OUTPUT_DIR = f"{DRIVE_ROOT}/output/qwen14b_t4"

os.makedirs(OUTPUT_DIR, exist_ok=True)

assert os.path.exists(TRAIN_PATH) and os.path.exists(VAL_PATH), (
    f"Upload train.jsonl and val.jsonl to {DRIVE_ROOT}/data/ in Google Drive first "
    "(from data/processed/hybrid/ in the repo)."
)

In [ ]:
from unsloth import FastLanguageModel
import torch

# T4 has only ~15GB usable VRAM and the 14B model + LoRA alone eats ~10GB of that
# before training starts - keep seq_len short to leave headroom for activations.
max_seq_length = 2048
dtype = torch.float16  # T4 is Turing - no fast native BF16 tensor cores, use FP16
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-14B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def format_chatml(examples):
    texts = [tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) for msg in examples["messages"]]
    return {"text": texts}

dataset_train = load_dataset("json", data_files=TRAIN_PATH, split="train")
dataset_val = load_dataset("json", data_files=VAL_PATH, split="train")

dataset_train = dataset_train.map(format_chatml, batched=True)
dataset_val = dataset_val.map(format_chatml, batched=True)

In [ ]:
from trl import SFTTrainer, SFTConfig

# batch_size=1 / grad_acc=32 (same effective batch 32 as the MI300X run) - T4's ~5GB
# of free headroom after the model load is too tight to risk a bigger per-step batch.
training_args = SFTConfig(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    per_device_eval_batch_size=1,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    packing=True,  # CUDA/T4 has working flash-attention via Unsloth - safe here, unlike our ROCm box
    logging_steps=10,
    report_to="tensorboard",
    logging_dir=f"{OUTPUT_DIR}/runs",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=50,
    output_dir=OUTPUT_DIR,
    optim="adamw_8bit",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset_train,
    eval_dataset=dataset_val,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=training_args,
)

In [ ]:
# Auto-resume: if this session got dropped and you're rerunning all cells,
# this picks up the latest checkpoint already saved on Drive instead of restarting.
last_checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    checkpoints = [d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")]
    if checkpoints:
        last_checkpoint = os.path.join(OUTPUT_DIR, sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1])
        print(f"Resuming from checkpoint: {last_checkpoint}")

trainer_stats = trainer.train(resume_from_checkpoint=last_checkpoint)

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Done! Adapter saved to {OUTPUT_DIR}")